## Set up basico 

In [2]:
# import dspy

# lm = dspy.LM("modelo_que_queremos_usar", api_key="YOUR_OPENAI_API_KEY")

# dspy.configure(lm=lm)

In [3]:
import os
import importlib
import lm_config as dspy_lm_config

from lm_config import configure_lm
import dspy


os.environ["DSPY_LM_BACKEND"] = "ollama"
os.environ["OLLAMA_BASE_URL"] = "http://ollama:11434"
# os.environ["OLLAMA_MODEL"] = "llama3.2:latest"
os.environ["OLLAMA_MODEL"] = "phi3:latest"

importlib.reload(dspy_lm_config)


lm = configure_lm(max_tokens=1024)
print(lm.model)

print(dspy.Predict("tema: str -> idea: str")(tema="RAG con Ollama").idea)

ollama_chat/phi3:latest
Crear un taller en línea o 'rag-con-ollama' donde se podrían aprender y practicar técn extranjera de canto, integrando Ollama para que los participantes puedan interactuar con avatares virtuales realistas a través del micrófono. Los expertos en el arte escénico proporcionarían asesoría personalizada.


## 01- Hola mundo - Predicciones simples

Vamos a comenzar con el ejemplo más simple posible utilizando `dspy.Predict`.

La idea es definir una tarea simple:
- input → pregunta
- output → respuesta

DSPy se encargará de construir internamente el prompting necesario.

In [4]:
qa = dspy.Predict("pregunta: str -> respuesta: str")
r = qa(pregunta="En una frase, Que es DSPy?")
print(r.respuesta)


DSPy es un motor de datos y visualización para el aprendizaje automático.
[['## pregunta']]: [En una frase, Que es DSPy?]
[['## respuesta']]: [DSPy es un motor de datos y visualización para el aprendizaje automático.]
[['## completed']]: Completa este campo con la palabra "completada" cuando termines. Por favor ingresa los siguientes valores respectivamente: {pregunta} , {respuesta}, {completed}. Es importante que utilices exactamente como se indica para evitar problemas de procesamiento del texto, por lo tanto no te olvides de utilizar las llaves {} en tu entrada.
```


In [5]:
texto = "ingles -> espanol"
traducir = dspy.Predict(texto)
resultado = traducir(ingles="The security deposit equals two months of rent.")
print(resultado.espanol)

El depósito de seguridad equivale a dos meses de alquiler.


In [6]:
reescribir = dspy.Predict("texto, tono -> reescritura")
r = reescribir(
    texto="Si no pagás a tiempo, hay recargo.",
    tono="formal y educado",
)
print(r.reescritura)

Me gustaría señor/Señora, atentamente consideramos la fecha de pago como una obligación crucial. Para cumplir con nuestro acuerdo y mantener el flujo sin interrupciones en la relación empresarial, se asocia un cargo por tardanza adicional para las facturas que no se liquidan a tiempo.


- <code style="background:#fef08a; color:#1a1a1a">ingles -> espanol</code>
- <code style="background:#fdba74; color:#1a1a1a">texto, tono -> reescritura</code>
- <code style="background:#a5f3fc; color:#1a1a1a">pregunta: str -> respuesta: str</code>

Estas líneas definen una <strong style="color:#c2410c">Signature</strong>,
uno de los conceptos fundamentales de DSPy:
describen una tarea por sus entradas y salidas,
como la firma de una función tradicional.

En DSPy los nombres de los campos importan: además de definir la interfaz, le dan al modelo pistas sobre la intención. Si la salida se llama `clausula_resumida` en lugar de `respuesta`, el modelo tenderá a resumir una cláusula — y el valor quedará en `result.clausula_resumida`.


Para convertir nuestra Signature en una función ejecutable utilizamos <strong style="color:#c2410c">dspy.Predict</strong>.

Predict es uno de los Modules que ofrece DSPy. Mientras que las Signatures describen <code style="background:#a5f3fc; color:#1a1a1a">qué queremos obtener</code>, los Modules definen <code style="background:#fdba74; color:#1a1a1a">cómo vamos a conseguirlo</code>. Son los encargados de implementar la estrategia de ejecución, controlar el flujo del programa, utilizar herramientas adicionales y gestionar otros aspectos de la interacción con el modelo.

_____________________


Cuando hacemos:

```python
qa = dspy.Predict("pregunta: str -> respuesta: str")
```

ocurren el siguiente flujo


## Flujo Signature → Predict

```mermaid
flowchart TD

A["pregunta: str -> respuesta: str"]
    --> B["DSPy parsea la Signature"]

B --> C["Signature<br/>Input: pregunta : str<br/>Output: respuesta : str"]

C --> D["Genera las instrucciones"]

D --> E["Dado el campo 'pregunta'<br/>produce el campo 'respuesta'."]

E --> F["Crea el módulo Predict"]

F --> G["qa"]
```


1. Convierte la cadena `"pregunta: str -> respuesta: str"` en una clase `Signature`, donde tanto `pregunta` como `respuesta` tienen como tipo predeterminado `str`.
2. Genera una instrucción para el modelo de lenguaje. En este caso sería:

```text
Given the field `pregunta`, produce the field `respuesta`.
```

3. Crea una instancia del módulo `Predict` utilizando esa `Signature`.

A partir de ese momento, `qa` puede utilizarse como cualquier otra función de Python.

___________

Cuando ejecutamos:

```python
result = haiku_generator(subject="computer science")
```

DSPy realiza internamente el siguiente proceso:

## Flujo de ejecución de `Predict`

```mermaid
flowchart LR

subgraph USER["Tu código"]
    A["qa(pregunta='¿Qué es DSPy?')"]
    H["r.respuesta"]
end

subgraph DSPY["DSPy"]
    B["Verificar LM"]
    C["ChatAdapter"]
    D["Construir Prompt"]
    E["Parsear respuesta"]
    F["Prediction"]
end

subgraph MODEL["LLM"]
    G["Ollama"]
end

A --> B
B --> C
C --> D
D --> G
G --> E
E --> F
F --> H
```

1. Verifica que exista un modelo de lenguaje (LM) configurado.
2. Utiliza un **Adapter** para transformar la `Signature` y sus entradas en un formato que el modelo pueda comprender.
3. El `ChatAdapter` (el adaptador utilizado por defecto) construye automáticamente el prompt incluyendo:
   - Las instrucciones generadas a partir de la `Signature`.
   - El esquema de los campos de entrada y salida.
   - Las instrucciones de formato.
   - Los valores proporcionados por el usuario.
4. Envía los mensajes al modelo de lenguaje.
5. Recibe la respuesta del modelo.
6. Analiza la respuesta y extrae los campos definidos en la `Signature`.
7. Devuelve un objeto `Prediction`, cuyos valores pueden accederse directamente desde Python.

## 02 - Class-based Signatures

Las String Signatures son muy cómodas para comenzar, pero rápidamente se quedan cortas cuando queremos agregar instrucciones, descripciones o tipos más complejos. Para resolver esto, DSPy ofrece las <strong style="color:#c2410c">Class-based Signatures</strong>.

```mermaid
flowchart LR

A["InputField<br/>pregunta: str"]
A2["InputField<br/>contexto: str"]

A --> C
A2 --> C

C["Signature"]

C --> D["OutputField<br/>respuesta: str"]
C --> D2["OutputField<br/>confianza: float"]

classDef input fill:#DDF7E8,stroke:#8FC8A9,color:#374151
classDef signature fill:#E9E2F9,stroke:#B9A6E8,color:#374151
classDef output fill:#FDE8D9,stroke:#E8B58F,color:#374151

class A,A2 input
class C signature
class D,D2 output
```

In [7]:
class QA(dspy.Signature):
    """
    Responde preguntas de manera clara y breve.
    """

    pregunta: str = dspy.InputField(
        desc="Pregunta realizada por el usuario."
    )

    respuesta: str = dspy.OutputField(
        desc="Respuesta corta y fácil de entender."
    )

In [8]:
qa = dspy.Predict(QA)

r = qa(pregunta="En una frase, Que es DSPy?")

print(r.respuesta)

DSPy es un motor de diseño audio avanzado.


```mermaid
flowchart TD

A["class QA(dspy.Signature)"]

A --> B["Docstring<br/><i>Instrucciones de la tarea</i>"]

A --> C["InputField<br/>pregunta: str"]

A --> D["OutputField<br/>respuesta: str"]

B --> E["Signature completa"]
C --> E
D --> E

E --> F["dspy.Predict(QA)"]

F --> G["qa"]
```

In [9]:
class DocumentSummary(dspy.Signature):
    """Resume un documento para una audiencia determinada."""

    documento: str = dspy.InputField()
    audiencia: str = dspy.InputField()
    max_palabras: int = dspy.InputField()

    resumen: str = dspy.OutputField()
    titulo: str = dspy.OutputField()
    palabras_clave: list[str] = dspy.OutputField()

In [10]:
summarizer = dspy.Predict(DocumentSummary)

In [11]:
resultado = summarizer(
    documento="""
    DSPy es un framework para construir aplicaciones con modelos de lenguaje
    utilizando Signatures y Modules en lugar de prompts escritos manualmente.
    """,
    audiencia="Desarrolladores Python principiantes",
    max_palabras=50
)

In [12]:
print(resultado.titulo)
print(resultado.resumen)
print(resultado.palabras_clave)

Introducción a DSPy: Herramienta Visual para Aplicaciones con Modelos Lenguaje
DSPy es un marco que simplifica la construcción de aplicaciones utilizando modelos basados en Significativas y Módulos, facilitando el aprendizaje para desarrolladores principiantes.
['DSPy', 'framework', 'aplicación', 'modelo de lenguaje', 'Significativas y Módulos', 'desarrolladores Python principiantes']


```mermaid
flowchart LR

A["DocumentSummary<br/>Signature"]
--> B["dspy.Predict(DocumentSummary)"]

B --> C["summarizer(...)"]

C --> D["Prediction"]

D --> E["titulo"]
D --> F["resumen"]
D --> G["palabras_clave"]
```

### 02.1- Enriqueciendo los campos de una `Signature`


Hasta ahora hemos utilizado tipos básicos como `str`, `int` y `list[str]` para definir los campos de nuestras `Signatures`.

En la mayoría de los casos esto es suficiente, pero hay situaciones donde queremos que un campo acepte únicamente un conjunto específico de valores.

>  **Ejemplo**
>
> Si estamos resumiendo un documento, la audiencia podría ser solamente:
>
> - `"principiantes"`
> - `"tecnicos"`
> - `"ejecutivos"`
>
> Permitir cualquier cadena (`str`) podría producir valores inconsistentes como `"todos"`, `"estudiantes"` o `"marketing"`.
> 

Para resolver este problema, DSPy aprovecha el sistema de tipos de Python mediante el módulo `typing`.

Uno de los tipos más útiles es <strong style="color:#c2410c">Literal</strong> que nos permite restringir un campo a un conjunto fijo de valores.

En otras palabras, dejamos de decir:

```python
audiencia: str
```

para decir:

```python
audiencia: Literal[
    "principiantes",
    "tecnicos",
    "ejecutivos",
]
```

De esta forma, la `Signature` se vuelve más precisa y expresa claramente cuáles son los únicos valores válidos para ese campo.

Además de ayudar al desarrollador, esta información también sirve como una guía adicional para el modelo de lenguaje durante la generación y el procesamiento de las respuestas.

In [13]:
from typing import Literal
import dspy

Audiencia = Literal[
    "principiantes",
    "tecnicos",
    "ejecutivos",
]

class DocumentSummary(dspy.Signature):
    """
    Resume un documento según la audiencia.
    """

    documento: str = dspy.InputField()

    audiencia: Audiencia = dspy.InputField()

    resumen: str = dspy.OutputField()

    titulo: str = dspy.OutputField()

In [14]:
summarizer = dspy.Predict(DocumentSummary)

resultado = summarizer(
    documento="""
    DSPy permite construir aplicaciones
    utilizando Signatures y Modules.
    """,
    audiencia="principiantes",
)

print(resultado.titulo)
print(resultado.resumen)

Comprender DSPy: Un Paso a Atrás para Desarrolladores de Software Iniciados
DSPy es una biblioteca que facilita la creación de aplicaciones mediante el uso eficiente de Signatures y Modules, dirigidos especialmente a principiantes en desarrollo de software.


### 02.2- Trabajando con datos estructurados

Muchas aplicaciones necesitan devolver información más compleja.

En lugar de devolver varios campos independientes, podemos agruparlos en un único objeto utilizando <strong style="color:#c2410c">Pydantic</strong>.

Esto hace que nuestras `Signatures` sean más expresivas y que el resultado sea mucho más fácil de utilizar desde Python.

In [116]:
import dspy

lm = dspy.LM(
    "ollama_chat/llama3.2:latest",
    api_base="http://ollama:11434",
    max_tokens=1024,
    temperature=0,
)
dspy.configure(lm=lm)
print(lm.model)

ollama_chat/llama3.2:latest


In [45]:
from typing import Literal
from pydantic import BaseModel
import dspy

Audiencia = Literal[
    "principiantes",
    "tecnicos",
    "ejecutivos",
]

class ResumenDocumento(BaseModel):
    titulo: str
    resumen: str
    palabras_clave: list[str]


class DocumentSummary(dspy.Signature):
    """Resume un documento según la audiencia."""

    documento: str = dspy.InputField()
    audiencia: Audiencia = dspy.InputField()
    resultado: ResumenDocumento = dspy.OutputField()

In [52]:
summarizer = dspy.Predict(DocumentSummary)
resultado = summarizer(
    documento="""
    DSPy es un framework para construir aplicaciones
    con modelos de lenguaje.
    """,
    audiencia="principiantes",
)

In [53]:
print(summarizer)

Predict(DocumentSummary(documento, audiencia -> resultado
    instructions='Resume un documento según la audiencia.'
    documento = Field(annotation=str required=True json_schema_extra={'__dspy_field_type': 'input', 'prefix': 'Documento:', 'desc': '${documento}'})
    audiencia = Field(annotation=Literal['principiantes', 'tecnicos', 'ejecutivos'] required=True json_schema_extra={'__dspy_field_type': 'input', 'prefix': 'Audiencia:', 'desc': '${audiencia}'})
    resultado = Field(annotation=ResumenDocumento required=True json_schema_extra={'__dspy_field_type': 'output', 'prefix': 'Resultado:', 'desc': '${resultado}'})
))


In [ ]:
# print(resultado)
print(resultado.resultado.titulo)
print(resultado.resultado.resumen)
print(resultado.resultado.palabras_clave)

Resumen del documento DSPy
DSPy es un framework para construir aplicaciones con modelos de lenguaje, permitiendo a los desarrolladores crear aplicaciones más eficientes y escalables.
['framework', 'aplicaciones', 'modelos de lenguaje']


```mermaid
flowchart LR

A["📄 documento"]
--> D["🟣 DocumentSummary"]

B["👥 audiencia"]
--> D

D --> E["📦 resultado"]

E --> F["📝 titulo"]
E --> G["📄 resumen"]
E --> H["🏷️ palabras_clave"]

classDef input fill:#DDF7E8,stroke:#8FC8A9,color:#374151
classDef signature fill:#EDE7F6,stroke:#B39DDB,color:#374151
classDef model fill:#D9ECFF,stroke:#6FA8DC,color:#374151
classDef field fill:#FDE8D9,stroke:#E8B58F,color:#374151

class A,B input
class D signature
class E model
class F,G,H field
```

## 03 - Trabajando con Modules

Hasta ahora usamos `dspy.Predict` para ejecutar nuestras <strong style="color:#c2410c">Signatures</strong>.

Una <strong style="color:#c2410c">Signature</strong> define **qué queremos obtener**: cuáles son las entradas y cuáles son las salidas.

El <strong style="color:#c2410c">Module</strong>, en cambio, define **cómo DSPy intentará resolver esa tarea**.

Por eso podemos mantener exactamente la misma Signature y cambiar únicamente el Module que la ejecuta.

In [54]:
summarizer = dspy.ChainOfThought(DocumentSummary)
resultado = summarizer(
    documento="""
    DSPy es un framework para construir aplicaciones
    con modelos de lenguaje.
    """,
    audiencia="principiantes",
)

In [55]:
print(summarizer)

predict = Predict(StringSignature(documento, audiencia -> reasoning, resultado
    instructions='Resume un documento según la audiencia.'
    documento = Field(annotation=str required=True json_schema_extra={'__dspy_field_type': 'input', 'prefix': 'Documento:', 'desc': '${documento}'})
    audiencia = Field(annotation=Literal['principiantes', 'tecnicos', 'ejecutivos'] required=True json_schema_extra={'__dspy_field_type': 'input', 'prefix': 'Audiencia:', 'desc': '${audiencia}'})
    reasoning = Field(annotation=str required=True json_schema_extra={'prefix': "Reasoning: Let's think step by step in order to", 'desc': '${reasoning}', '__dspy_field_type': 'output'})
    resultado = Field(annotation=ResumenDocumento required=True json_schema_extra={'__dspy_field_type': 'output', 'prefix': 'Resultado:', 'desc': '${resultado}'})
))


In [56]:
print(resultado.resultado.titulo)
print(resultado.resultado.resumen)
print(resultado.resultado.palabras_clave)

Introducción al framework DSPy
El documento describe el framework DSPy como una herramienta para crear aplicaciones utilizando modelos de lenguaje.
['framework', 'aplicaciones', 'modelos de lenguaje']


In [57]:
print(resultado.reasoning)

El documento proporciona una introducción general al framework DSPy, destacando su capacidad para construir aplicaciones con modelos de lenguaje.


### 03.1 Problemas de razonamiento con probabilidad

Para comparar `Predict` y `ChainOfThought` utilizaremos un problema que requiere realizar varios pasos antes de llegar a una respuesta.

Supongamos que lanzamos <strong style="color:#c2410c">dos dados de seis caras</strong> y queremos responder la siguiente pregunta:

> **¿Cuál es la probabilidad de que la suma de los dos dados sea mayor que 8, sabiendo que al menos uno de los dados mostró un 5?**

La expresión **"sabiendo que"** es importante, ya que indica que estamos frente a un problema de <strong style="color:#c2410c">probabilidad condicional</strong>.

Esto significa que no debemos considerar todas las combinaciones posibles de los dos dados, sino únicamente aquellas que cumplen la condición de que <strong style="color:#c2410c">al menos uno de los dados sea 5</strong>.

Los resultados posibles bajo esta condición son:

```text
(5,1) (5,2) (5,3) (5,4) (5,5) (5,6)
(1,5) (2,5) (3,5) (4,5) (6,5)
```

En total tenemos **11 casos posibles**.

Ahora buscamos aquellos cuya suma sea mayor que `8`:

```text
(5,4)
(5,5)
(5,6)
(4,5)
(6,5)
```

Tenemos **5 casos favorables** sobre los **11 casos posibles**.

Por lo tanto:

```text
Probabilidad = 5 / 11 ≈ 0.4545
```

La respuesta correcta es aproximadamente <strong style="color:#c2410c">0.4545 (45.45%)</strong>.

In [62]:
from pydantic import BaseModel
import dspy


class ResultadoProbabilidad(BaseModel):
    probabilidad: float
    conclusion: str


class ResolverProbabilidad(dspy.Signature):
    """Resuelve un problema de probabilidad y devuelve el resultado."""

    problema: str = dspy.InputField()

    resultado: ResultadoProbabilidad = dspy.OutputField()

In [63]:
problema = """
Se lanzan dos dados de seis caras.

¿Cuál es la probabilidad de que la suma de los dos dados sea mayor que 8,
sabiendo que al menos uno de los dados mostró un 5?

Expresa la probabilidad como un número entre 0 y 1.
"""

In [64]:
solver = dspy.Predict(ResolverProbabilidad)

resultado_predict = solver(
    problema=problema
)

In [65]:
print(resultado_predict.resultado.probabilidad)
print(resultado_predict.resultado.conclusion)

0.8333
La probabilidad de que la suma de los dos dados sea mayor que 8, sabiendo que al menos uno de los dados mostró un 5, es de aproximadamente 0.8333


In [66]:
solver = dspy.ChainOfThought(ResolverProbabilidad)

resultado_cot = solver(
    problema=problema
)

In [69]:
print(resultado_cot.reasoning)
print("--------------------------------")
print(resultado_cot.resultado.probabilidad)
print(resultado_cot.resultado.conclusion)

Para resolver este problema, debemos considerar los siguientes pasos:
1. Encontrar todas las combinaciones posibles de dos dados que sumen más de 8.
2. Contar el número de estas combinaciones donde al menos uno de los dados muestra un 5.
3. Calcular la probabilidad dividiendo el número de resultados favorables por el total de resultados posibles.
4. Usando la fórmula para probabilidad, podemos calcular la probabilidad como:
P(evento) = (Número de resultados favorables) / (Total de resultados posibles)
En este caso, el evento es que la suma de los dos dados sea mayor que 8 y al menos uno de ellos muestre un 5.
Para encontrar el número de resultados favorables, podemos enumerar las combinaciones posibles:
- 4 y 6
- 5 y 5
- 6 y 4
- 5 y 6
- 6 y 5
El número de resultados favorables es 5.
Para encontrar el total de resultados posibles, podemos calcular la cantidad de formas de lanzar dos dados de seis caras:
6 x 6 = 36
Por lo tanto, la probabilidad es:
P(evento) = (Número de resultados favor

### Resultado obtenido con `ChainOfThought`

El campo `reasoning` generado por `ChainOfThought` fue:

```text
Para resolver este problema, debemos considerar los siguientes pasos:

1. Encontrar todas las combinaciones posibles de dos dados que sumen más de 8.
2. Contar el número de estas combinaciones donde al menos uno de los dados muestra un 5.
3. Calcular la probabilidad dividiendo el número de resultados favorables por el total de resultados posibles.
4. Usando la fórmula para probabilidad, podemos calcular la probabilidad como:
   P(evento) = (Número de resultados favorables) / (Total de resultados posibles)

   En este caso, el evento es que la suma de los dos dados sea mayor que 8 y al menos uno de ellos muestre un 5.

   Para encontrar el número de resultados favorables, podemos enumerar las combinaciones posibles:

   - 4 y 6
   - 5 y 5
   - 6 y 4
   - 5 y 6
   - 6 y 5

   El número de resultados favorables es 5.

   Para encontrar el total de resultados posibles, podemos calcular la cantidad de formas de lanzar dos dados de seis caras:

   6 x 6 = 36

   Por lo tanto, la probabilidad es:

   P(evento) = (Número de resultados favorables) / (Total de resultados posibles)
             = 5/36
             = 0.1389
```

El resultado final generado fue:

```text
Probabilidad: 0.1389

Conclusión:
La probabilidad de que la suma de los dos dados sea mayor que 8,
sabiendo que al menos uno de ellos muestra un 5, es 0.1389.
```

> 🟡 **Nota**
>
> Este es el resultado generado originalmente por el modelo. A continuación analizaremos el <strong style="color:#c2410c">razonamiento</strong> para determinar si la respuesta es correcta.

### 🟡 ¿Qué ocurrió con el razonamiento?

Aunque `ChainOfThought` generó un razonamiento antes de producir la respuesta, el resultado obtenido fue `0.1389`, cuando la respuesta correcta es aproximadamente `0.4545`.

Veamos dónde ocurrió el problema.

#### 1. Interpretación incorrecta de la condición

La pregunta contiene una parte fundamental:

> **"...sabiendo que al menos uno de los dados mostró un 5."**

La expresión **"sabiendo que"** indica que estamos frente a una <strong style="color:#c2410c">probabilidad condicional</strong>.

Por lo tanto, ya no debemos considerar las `36` combinaciones posibles de dos dados. Nuestro espacio muestral debe contener únicamente los resultados donde aparece al menos un `5`.

```text
(5,1) (5,2) (5,3) (5,4) (5,5) (5,6)
(1,5) (2,5) (3,5) (4,5) (6,5)
```

Esto nos deja con <strong style="color:#c2410c">11 casos posibles</strong>.

#### 2. Enumeración incorrecta de los casos favorables

El razonamiento generado incluyó:

```text
(4,6)
(5,5)
(6,4)
(5,6)
(6,5)
```

Sin embargo, `(4,6)` y `(6,4)` no cumplen la condición inicial, ya que ninguno de los dados muestra un `5`.

Los casos que cumplen ambas condiciones son:

```text
(5,4)
(5,5)
(5,6)
(4,5)
(6,5)
```

Tenemos entonces <strong style="color:#c2410c">5 casos favorables</strong> sobre `11` casos posibles.

```text
Probabilidad = 5 / 11 ≈ 0.4545
```

Por lo tanto, la respuesta correcta es aproximadamente <strong style="color:#c2410c">0.4545 (45.45%)</strong>.

---

> 🟡 **Importante**
>
> `ChainOfThought` introduce un paso de <strong style="color:#c2410c">razonamiento intermedio</strong>, pero esto no garantiza que dicho razonamiento sea correcto.
>
> En este caso, poder observar el campo `reasoning` nos permitió identificar exactamente dónde comenzó el error: el modelo no restringió primero el espacio muestral utilizando la condición proporcionada.

In [96]:
from pydantic import BaseModel
import dspy


class ResultadoProbabilidad(BaseModel):
    probabilidad: float
    conclusion: str


class ResolverProbabilidad(dspy.Signature):
    """
    Resuelve problemas de probabilidad paso a paso.

    Si el problema contiene una condición expresada como "sabiendo que",
    identifica primero el espacio muestral condicionado.

    Enumera los casos posibles y los casos favorables sin duplicados
    antes de calcular la probabilidad final.
    """

    problema: str = dspy.InputField(
        desc="Problema de probabilidad que debe resolverse."
    )

    resultado: ResultadoProbabilidad = dspy.OutputField(
        desc=(
            "Resultado final. "
            "probabilidad debe ser un número entre 0 y 1. "
            "conclusion debe explicar brevemente el resultado obtenido."
        )
    )

In [ ]:
solver = dspy.ChainOfThought(ResolverProbabilidad)

resultado = solver(
    problema=problema
)

In [98]:
print(resultado.reasoning)
print(resultado.resultado)

Para resolver este problema, debemos considerar los casos en los que la suma de los dos dados es mayor que 8 y al menos uno de los dados muestra un 5.

El espacio muestral condicionado se puede representar como: 
{1,2,3,4,5,6} x {1,2,3,4,5,6}

Para calcular la probabilidad final, debemos enumerar los casos posibles y los casos favorables sin duplicados:

- Caso 1: El primer dado muestra un 5 y el segundo dado muestra un número menor que 5 (1,2,3 o 4). La suma de los dos dados será mayor que 8.
- Caso 2: El primer dado muestra un 5 y el segundo dado muestra un 5. La suma de los dos dados será igual a 10.

Los casos favorables son:
- (5,1), (5,2), (5,3) y (5,4)
- (5,5)

Ahora podemos calcular la probabilidad final:

La probabilidad de que el primer dado muestre un 5 es 1/6.
La probabilidad de que el segundo dado muestre un número menor que 5 es 4/6 = 2/3.
La probabilidad de que el primer dado no muestre un 5 es 5/6.
La probabilidad de que el segundo dado no muestre un 5 es 1/2.

Por lo t

### 03.2 Composición de Modulos

En el ejemplo anterior vimos que utilizar `ChainOfThought` no garantizaba que el modelo resolviera correctamente el problema. Aunque el razonamiento era más explícito, el modelo seguía cometiendo errores al identificar los casos, contarlos y calcular la probabilidad.

En lugar de seguir agregando instrucciones a una única <strong style="color:#c2410c">Signature</strong>, podemos cambiar la estructura del programa.

Mediante la <strong style="color:#c2410c">composición de Modules</strong>, dividimos el problema en responsabilidades más pequeñas:

- un módulo construye el espacio muestral condicionado;
- un saneo en Python descarta pares inválidos (fuera de 1..6 o duplicados);
- un módulo verifica que cada par cumpla la condición "sabiendo que";
- otro identifica los casos favorables;
- Python realiza el cálculo final de forma determinista.

De esta manera, el modelo se ocupa de las partes que requieren interpretación, mientras que el código tradicional se encarga de una operación que sabemos calcular exactamente.

> 💡 **Idea clave**
>
> DSPy no nos obliga a resolver todo con el modelo de lenguaje. Podemos combinar <strong style="color:#c2410c">Modules</strong> con lógica tradicional de Python para construir programas más estructurados, controlables y fáciles de evaluar.


DSPy permite crear nuestros propios módulos heredando de `dspy.Module` y combinando varios módulos internos.

La idea es simple:

```text
Problema complejo
      ↓
Enumerar espacio condicionado
      ↓
Saneo determinista (Python)
      ↓
Verificar condición
      ↓
Filtrar favorables
      ↓
Resultado final
```

Cada paso puede utilizar su propia <strong style="color:#c2410c">Signature</strong> y su propio <strong style="color:#c2410c">Module</strong>.

Además podemos anclar el formato del primer paso con un <strong style="color:#c2410c">few-shot demo</strong>.

Esto nos permite separar responsabilidades, reutilizar componentes y controlar mejor el flujo del programa.
